In [1]:
import torch
import torch.nn as nn

# ---------------------------
# 1) 우리가 풀 예제 데이터: XOR
# ---------------------------
# X: (샘플 수, 입력 차원) = (4, 2)
# 각 행이 하나의 샘플이고, 열 2개가 x1, x2
X = torch.tensor([
    [0.0, 0.0],  # (x1, x2)
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0]
])

# y: (샘플 수, 출력 차원) = (4, 1)
y = torch.tensor([
    [0.0],
    [1.0],
    [1.0],
    [0.0]
])

# ---------------------------------------------------------
# 2) nn.Module이 뭔가?
#    - "모델 하나"를 나타내는 PyTorch의 표준 클래스
#    - 레이어(Linear 등)들을 등록해두면
#      model.parameters()로 W,b를 자동 수집 가능
#    - forward()에서 계산 흐름을 정의
# ---------------------------------------------------------
class MyXorMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # super().__init__()는 nn.Module 초기화를 위해 필수

        # -------------------------------------------------
        # 3) 레이어 정의 (가중치가 있는 층)
        #    입력 2 -> 히든 2 -> 출력 1 구조
        #
        #    hidden = Linear(2, 2)
        #      - 입력 차원 2개(x1,x2)
        #      - 출력(히든 노드) 2개(h1,h2)
        #
        #    out = Linear(2, 1)
        #      - 히든 2개를 받아 출력 1개(ŷ) 생성
        # -------------------------------------------------
        self.hidden = nn.Linear(2, 2, bias=True)
        self.out = nn.Linear(2, 1, bias=True)

        # -----------------------------------------
        # 4) "학습 없이" 우리가 원하는 직선(a,b)을
        #    직접 만들기 위해 가중치/편향을 수동 설정
        #
        # hidden이 만드는 식:
        #   [z1]   [w11 w12] [x1]   [b1]
        #   [z2] = [w21 w22] [x2] + [b2]
        #
        # 그리고 ReLU를 통과:
        #   h1 = ReLU(z1)
        #   h2 = ReLU(z2)
        #
        # 우리가 원하는 건:
        #   z1 = x1 - x2  (직선 기준: x1 - x2 = 0)
        #   z2 = x2 - x1  (직선 기준: x2 - x1 = 0)
        #
        # 그래서 W를 다음처럼 둔다:
        #   [ 1  -1]
        #   [-1   1]
        # b는 0
        # -----------------------------------------
        with torch.no_grad():
            self.hidden.weight[: ] = torch.tensor([
                [ 1.0, -1.0],  # z1 = 1*x1 + (-1)*x2
                [-1.0,  1.0]   # z2 = (-1)*x1 + 1*x2
            ])
            self.hidden.bias[:] = torch.tensor([0.0, 0.0])

            # -----------------------------------------
            # 출력층은 h1 + h2를 만들어주면 됨.
            # out의 식: ŷ = w1*h1 + w2*h2 + b
            # 여기서 (w1,w2) = (1,1), b=0
            # -----------------------------------------
            self.out.weight[:] = torch.tensor([[1.0, 1.0]])
            self.out.bias[:] = torch.tensor([0.0])

        # ReLU는 파라미터가 없는 "함수"라서 이렇게 만들어도 되고
        self.relu = nn.ReLU()

    # -----------------------------------------
    # 5) forward(): 모델이 "어떻게 계산하는지" 정의
    #    model(X)를 호출하면 내부에서 forward(X)가 실행됨
    # -----------------------------------------
    def forward(self, x):
        # (1) 선형 변환: z = xW^T + b
        z = self.hidden(x)

        # (2) 비선형 활성화: h = ReLU(z)
        h = self.relu(z)

        # (3) 출력 선형 변환: y_hat = out(h)
        y_hat = self.out(h)

        return y_hat, h, z  # 중간값들도 보고 싶어서 같이 반환


# ---------------------------
# 6) 모델 생성 및 실행
# ---------------------------
model = MyXorMLP()

with torch.no_grad():
    y_hat, h, z = model(X)

print("입력 X:\n", X)
print("\n히든층 선형 출력 z (=직선 a,b에 대한 점수):\n", z)
print("\n히든층 활성화 출력 h (=0/1처럼 쓰는 표현공간):\n", h)
print("\n최종 출력 y_hat:\n", y_hat)
print("\n최종 출력 반올림(round):\n", y_hat.round())
print("\n정답 y:\n", y)


입력 X:
 tensor([[0., 0.],
        [0., 1.],
        [1., 0.],
        [1., 1.]])

히든층 선형 출력 z (=직선 a,b에 대한 점수):
 tensor([[ 0.,  0.],
        [-1.,  1.],
        [ 1., -1.],
        [ 0.,  0.]])

히든층 활성화 출력 h (=0/1처럼 쓰는 표현공간):
 tensor([[0., 0.],
        [0., 1.],
        [1., 0.],
        [0., 0.]])

최종 출력 y_hat:
 tensor([[0.],
        [1.],
        [1.],
        [0.]])

최종 출력 반올림(round):
 tensor([[0.],
        [1.],
        [1.],
        [0.]])

정답 y:
 tensor([[0.],
        [1.],
        [1.],
        [0.]])


In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.5.1
True
